<a href="https://colab.research.google.com/github/AbeeraImran/GEN-AI_Seq2Seq_UrduQA/blob/MuhammadAhmad/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install datasets sentencepiece sacrebleu rouge-score torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=f7f288a079b1d0534e81b1ccac36c539dbdeb3802b9409a0cd872fdd2718ede8
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [3]:
from datasets import load_dataset

ds = load_dataset("uqa/UQA")

ex = ds["train"][0]
print("Keys:", ex.keys())
print("Question:", ex["question"])
print("Answer Data:", ex.get("answer") or ex.get("answers"))

n_total = len(ds["train"])
n_ans = sum(not a["is_impossible"] for a in ds["train"])
print(f"train rows: {n_total}, answerable: {n_ans}")

README.md:   0%|          | 0.00/898 [00:00<?, ?B/s]

data/train-00000-of-00001-bac007e8ca7192(…): reconstructing file:   0%|          |  0.00B / 30.2MB            

data/train-00000-of-00001-bac007e8ca7192(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-cf8a6960d(…): reconstructing file:   0%|          |  0.00B / 2.92MB            

data/validation-00000-of-00001-cf8a6960d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/124745 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/16824 [00:00<?, ? examples/s]

Keys: dict_keys(['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'])
Question: بیونس نے کب مقبولیت حاصل کرنا شروع کی؟
Answer Data: 1990 کی دہائی کے آخر میں
train rows: 124745, answerable: 83018


In [4]:
import csv

ANS_OPEN, ANS_CLOSE = "<ans>", "</ans>"
SENT_DELIMS = "\u06D4\u061F!"

def split_sentences(text):
    start = 0
    for i, ch in enumerate(text):
        if ch in SENT_DELIMS:
            yield start, i + 1, text[start:i + 1]
            start = i + 1
    if start < len(text):
        yield start, len(text), text[start:]

def make_pair(example, max_src=60, max_tgt=25):
    if example.get("is_impossible", False):
        return None

    ans_data = example.get("answer") or example.get("answers")
    start_data = example.get("answer_start")

    try:
        if isinstance(ans_data, dict):
            a_text = ans_data["text"][0]
            a_start = ans_data["answer_start"][0]
        else:
            a_text = ans_data[0] if isinstance(ans_data, list) else ans_data
            a_start = start_data[0] if isinstance(start_data, list) else start_data
    except (IndexError, KeyError, TypeError):
        return None

    context = example["context"]

    for s, e, sent in split_sentences(context):
        if s <= a_start < e:
            rel = a_start - s
            if sent[rel:rel + len(a_text)] != a_text:
                return None

            src = (sent[:rel] + " " + ANS_OPEN + " " + a_text + " "
                   + ANS_CLOSE + " " + sent[rel + len(a_text):]).strip()
            src = " ".join(src.split())
            tgt = " ".join(example["question"].split())

            if len(src.split()) > max_src or len(tgt.split()) > max_tgt:
                return None
            return src, tgt
    return None

def build_split(split, out_path):
    pairs = [p for p in map(make_pair, split) if p is not None]
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        w.writerows(pairs)
    print(f"{out_path}: {len(pairs)} pairs")
    return pairs

train_pairs = build_split(ds["train"], "train.tsv")
valid_pairs = build_split(ds["validation"], "valid.tsv")

train.tsv: 75067 pairs
valid.tsv: 10018 pairs


In [5]:
print("source ",train_pairs[0][0])
print("target ",train_pairs[0][1])

source  ہیوسٹن ، ٹیکساس میں پیدا ہوئی اور اس کی پرورش ہوئی ، اس نے بچپن میں مختلف گانے اور رقص کے مقابلوں میں پرفارم کیا ، اور <ans> 1990 کی دہائی کے آخر میں </ans> R&B گرل گروپ ڈسٹنی چائلڈ کے لیڈ گلوکار کی حیثیت سے شہرت حاصل کی۔
target  بیونس نے کب مقبولیت حاصل کرنا شروع کی؟


In [6]:
import sentencepiece as spm

ANS_OPEN, ANS_CLOSE = "<ans>", "</ans>"

with open("sp_corpus.txt", "w", encoding="utf-8") as f:
    for src, tgt in train_pairs:
        f.write(src + "\n" + tgt + "\n")

spm.SentencePieceTrainer.train(
    input="sp_corpus.txt",
    model_prefix="ur_sp",
    vocab_size=8000,
    model_type="unigram",
    character_coverage=1.0,
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE],
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)

sp = spm.SentencePieceProcessor(model_file="ur_sp.model")
PAD, UNK, BOS, EOS = 0, 1, 2, 3

src, tgt = train_pairs[0]
print("Tokens:", sp.encode(src, out_type=str))
print("IDs:", sp.encode(tgt))
print("Match Check:", sp.decode(sp.encode(tgt)) == tgt)

Tokens: ['▁ہیوسٹن', '▁،', '▁ٹیکساس', '▁میں', '▁پیدا', '▁ہوئی', '▁اور', '▁اس', '▁کی', '▁پرورش', '▁ہوئی', '▁،', '▁اس', '▁نے', '▁بچپن', '▁میں', '▁مختلف', '▁گانے', '▁اور', '▁رقص', '▁کے', '▁مقابلوں', '▁میں', '▁پرفارم', '▁کیا', '▁،', '▁اور', '▁', '<ans>', '▁1990', '▁کی', '▁دہائی', '▁کے', '▁آخر', '▁میں', '▁', '</ans>', '▁R', '&', 'B', '▁گر', 'ل', '▁گروپ', '▁ڈسٹنی', '▁چائلڈ', '▁کے', '▁لیڈ', '▁گلوکار', '▁کی', '▁حیثیت', '▁سے', '▁شہرت', '▁حاصل', '▁کی۔']
IDs: [2757, 18, 83, 2810, 100, 151, 99, 9, 11]
Match Check: True


Dataset class

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class QGDataset(Dataset):
    def __init__(self, tsv_path, sp_model):
        self.sp = sp_model
        self.pairs = []

        with open(tsv_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue

                parts = line.split('	')
                if len(parts) == 2:
                    self.pairs.append((parts[0], parts[1]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]

        src_ids = self.sp.encode(src_text)

        tgt_ids = [2] + self.sp.encode(tgt_text) + [3]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

In [8]:
train_dataset = QGDataset("train.tsv", sp)
print("Total dataset size:", len(train_dataset))
src_sample, tgt_sample = train_dataset[0]
print("Source tensor:", src_sample)
print("Target tensor:", tgt_sample)

Total dataset size: 75067
Source tensor: tensor([1159,   10, 2741,    8,  192,  144,   12,   21,    9, 5942,  144,   10,
          21,   18, 3923,    8,  166, 1224,   12, 3207,    7, 4386,    8, 2612,
          16,   10,   12,    6,    4, 1091,    9,  168,    7,  324,    8,    6,
           5, 1331, 6940, 1330,  751,  103,  198, 5159, 4084,    7, 2412, 3092,
           9,  403,   13, 5250,  100,  175])
Target tensor: tensor([   2, 2757,   18,   83, 2810,  100,  151,   99,    9,   11,    3])


In [9]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Subset

def collate_fn(batch):
    src_list = [item[0] for item in batch]
    tgt_list = [item[1] for item in batch]
    src_lengths = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    padded_src = pad_sequence(src_list, batch_first=True, padding_value=0)
    padded_tgt = pad_sequence(tgt_list, batch_first=True, padding_value=0)
    return padded_src, padded_tgt, src_lengths

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

valid_dataset = QGDataset("valid.tsv", sp)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [10]:
batch=next(iter(train_loader))

src,tgt,lengths=batch

print("source shape ",src.shape)
print("target shape ",tgt.shape)
print("lengths shape ",lengths.shape)
print("first source length ",lengths[0])
print("first source ",src[0])
print("first target ",tgt[0])

source shape  torch.Size([64, 95])
target shape  torch.Size([64, 28])
lengths shape  torch.Size([64])
first source length  tensor(28)
first source  tensor([1621,  183,    8,   10,   21,   97,   31,   12, 5354,   31,   15,    6,
           4, 1382,    6,    5, 3906,    9,   50,   51,    8,   13,   20,    8,
         270,   16,   38,   25,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0])
first target  tensor([   2, 1621,  183,    8,   21,   97,   31,   12, 5354,   31,    8,   92,
        3906,    7, 1393,    8,  427,    9, 1798,   14,   11,    3,    0,    0,
           0,    0,    0,    0])


Encoder:

In [11]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Encoder(nn.Module):
    def __init__(self, vocab_size=8000, emb_dim=256, hidden_size=512, num_layers=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout,
            batch_first=True
        )

    def forward(self, src, src_lengths):
        embedded = self.embedding(src)

        packed = pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        outputs, (hidden, cell) = self.rnn(packed)

        #outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        outputs,_=pad_packed_sequence(
            outputs,batch_first=True,
            total_length=src.size(1)
        )
        return outputs, hidden, cell

In [12]:
encoder = Encoder()

batch_src, batch_tgt, batch_lens = next(iter(train_loader))

enc_outputs, enc_hidden, enc_cell = encoder(batch_src, batch_lens)

print("Encoder Output Shape:", enc_outputs.shape)
print("Hidden State Shape:", enc_hidden.shape)
print("Cell State Shape:", enc_cell.shape)

Encoder Output Shape: torch.Size([64, 73, 1024])
Hidden State Shape: torch.Size([4, 64, 512])
Cell State Shape: torch.Size([4, 64, 512])


Decoder:

In [16]:
class BahdanauAttention(nn.Module):
    def __init__(self,encoder_hidden_size=512, decoder_hidden_size=512, attention_dim=256):
        super().__init__()

        encoder_output_dim=encoder_hidden_size*2

        self.encoder_projection=nn.Linear(
            encoder_output_dim,
            attention_dim
        )

        self.decoder_projection=nn.Linear(
            decoder_hidden_size,
            attention_dim
        )

        self.energy=nn.Linear(
            attention_dim,
            1
        )

    def forward(self,decoder_hidden,encoder_outputs,mask=None):

        encoder_features=self.encoder_projection(encoder_outputs)
        decoder_features=self.decoder_projection(decoder_hidden)

        decoder_features=decoder_features.unsqueeze(1)

        combined=torch.tanh(
                encoder_features+decoder_features
            )

        energy=self.energy(combined).squeeze(2)

        if mask is not None:
                energy=energy.masked_fill(~mask,-1e10)

        attention_weights=torch.softmax(energy,dim=1)
        context=torch.bmm(
                attention_weights.unsqueeze(1),
                encoder_outputs
                ).squeeze(1)
        return context,attention_weights


class AttentionDecoder(nn.Module):
    def __init__(
        self,
        vocab_size=8000,
        emb_dim=256,
        hidden_size=512,
        num_layers=2,
        dropout=0.3,
        attention_dim=256
    ):
        super().__init__()

        self.embedding=nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=0
        )

        self.attention=BahdanauAttention(
            encoder_hidden_size=hidden_size,
            decoder_hidden_size=hidden_size,
            attention_dim=attention_dim
        )

        self.rnn=nn.LSTM(
            input_size=emb_dim+(hidden_size *2),
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )

        self.out=nn.Linear(
            hidden_size+(hidden_size*2),
            vocab_size
        )

        self.dropout=nn.Dropout(dropout)

    def forward(self,input_token,hidden,cell,encoder_outputs,mask=None):
        if input_token.dim()==2:
          input_token=input_token.squeeze(1)
        input_token=input_token.unsqueeze(1)
        embedded=self.embedding(input_token)
        embedded=self.dropout(embedded)

        decoder_hidden=hidden[-1]

        context,attention_weights=self.attention(
                decoder_hidden,
                encoder_outputs,
                mask
            )

        context=context.unsqueeze(1)
        rnn_input=torch.cat(
                (embedded,context),
                dim=2
            )

        output,(hidden,cell)=self.rnn(
                rnn_input,
                (hidden,cell)
            )

        output=output.squeeze(1)

        context=context.squeeze(1)

        prediction_input=torch.cat(
                (output,context),
                dim=1
            )

        prediction=self.out(prediction_input)
        return prediction,hidden,cell,attention_weights




training loop

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder().to(device)
decoder = AttentionDecoder().to(device)

params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = optim.Adam(params, lr=0.001)

criterion = nn.CrossEntropyLoss(ignore_index=0)

TEACHER_FORCING_RATIO = 0.5

def train_one_epoch(encoder, decoder, loader, optimizer, criterion):
    encoder.train()
    decoder.train()
    total_loss = 0

    for src, tgt, src_lengths in loader:
        src, tgt, src_lengths = src.to(device), tgt.to(device), src_lengths.to(device)
        optimizer.zero_grad()

        encoder_outputs, hidden, cell = encoder(src, src_lengths)

        batch_size, tgt_len = tgt.shape

        hidden = hidden.view(2, 2, batch_size, 512)
        hidden = hidden[:, 0, :, :] + hidden[:, 1, :, :]

        cell = cell.view(2, 2, batch_size, 512)
        cell = cell[:, 0, :, :] + cell[:, 1, :, :]

        src_mask=(torch.arange(
            src.size(1),
            device=device
        ).unsqueeze(0)<src_lengths.unsqueeze(1))

        input_token=tgt[:,0].unsqueeze(1)
        loss = 0

        for t in range(1, tgt_len):
            prediction, hidden, cell,attention_weights = decoder(input_token, hidden, cell,encoder_outputs,src_mask)
            loss += criterion(prediction, tgt[:, t])

            if random.random() < TEACHER_FORCING_RATIO:
                input_token = tgt[:, t].unsqueeze(1)
            else:
                input_token = prediction.argmax(1).unsqueeze(1)

        loss = loss / (tgt_len - 1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters())+list(decoder.parameters()),
            max_norm=1.0
        )
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [18]:
import torch

EPOCHS = 15
best_valid_loss = float('inf')
train_losses=[]
valid_losses=[]

def evaluate_loss(encoder, decoder, valid_loader, criterion):
    encoder.eval()
    decoder.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt, src_lengths in valid_loader:
            src, tgt = src.to(device), tgt.to(device)
            src_lengths=src_lengths.to(device)

            encoder_outputs, hidden, cell = encoder(src, src_lengths)

            batch_size, tgt_len = tgt.shape
            hidden = hidden.view(2, 2, batch_size, 512)
            hidden = hidden[:, 0, :, :] + hidden[:, 1, :, :]
            cell = cell.view(2, 2, batch_size, 512)
            cell = cell[:, 0, :, :] + cell[:, 1, :, :]

            src_mask=(
                torch.arange(
                    src.size(1),
                    device=device
                ).unsqueeze(0)<src_lengths.unsqueeze(1)
            )

            input_token = tgt[:, 0].unsqueeze(1)
            loss = 0

            for t in range(1, tgt_len):
                prediction, hidden, cell,attention_weights = decoder(input_token, hidden, cell,encoder_outputs,src_mask)
                loss += criterion(prediction, tgt[:, t])
                input_token = tgt[:, t].unsqueeze(1)

            loss = loss / (tgt_len - 1)
            epoch_loss += loss.item()

    return epoch_loss / len(valid_loader)

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(encoder, decoder, train_loader, optimizer, criterion)
    valid_loss = evaluate_loss(encoder, decoder, valid_loader, criterion)
    train_losses.append(train_loss)
    valid_losses.append(valid_losses)

    print(f"Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print("Saving new best checkpoint!")
        torch.save({
            'encoder_state_dict': encoder.state_dict(),
            'decoder_state_dict': decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch':epoch+1,
            'train_loss':train_loss,
            'loss':valid_loss
        }, 'best_seq2seq_model.pt')

Epoch: 01 | Train Loss: 4.787 | Val Loss: 4.064
Saving new best checkpoint!
Epoch: 02 | Train Loss: 4.358 | Val Loss: 3.827
Saving new best checkpoint!
Epoch: 03 | Train Loss: 4.138 | Val Loss: 3.635
Saving new best checkpoint!
Epoch: 04 | Train Loss: 3.926 | Val Loss: 3.539
Saving new best checkpoint!
Epoch: 05 | Train Loss: 3.680 | Val Loss: 3.479
Saving new best checkpoint!
Epoch: 06 | Train Loss: 3.440 | Val Loss: 3.441
Saving new best checkpoint!
Epoch: 07 | Train Loss: 3.241 | Val Loss: 3.446
Epoch: 08 | Train Loss: 3.071 | Val Loss: 3.461
Epoch: 09 | Train Loss: 2.933 | Val Loss: 3.511
Epoch: 10 | Train Loss: 2.806 | Val Loss: 3.480
Epoch: 11 | Train Loss: 2.708 | Val Loss: 3.492
Epoch: 12 | Train Loss: 2.598 | Val Loss: 3.541
Epoch: 13 | Train Loss: 2.529 | Val Loss: 3.544
Epoch: 14 | Train Loss: 2.445 | Val Loss: 3.556
Epoch: 15 | Train Loss: 2.367 | Val Loss: 3.612


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss")
plt.plot(range(1, len(valid_losses) + 1), valid_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.savefig("training_validation_loss.png", dpi=300, bbox_inches="tight")
plt.show()

In [19]:
import os

print(os.path.exists("best_seq2seq_model.pt"))

True


In [20]:
checkpoint=torch.load('best_seq2seq_model.pt',
                      map_location=device

                      )
encoder.load_state_dict(checkpoint["encoder_state_dict"])
decoder.load_state_dict(checkpoint["decoder_state_dict"])

encoder.eval()
decoder.eval()
print("model loaded")

model loaded
